## Encoding and Transformer

### Encoder MLP
From observation data ( which includes observations and unseen data ),  
We put the data $x$ into the encoder $q_{\phi}(y \mid o )$, then sample "latent data vector" $y^{(i)} \sim q_{\phi}(y \mid o )$ .

Where, we model the distribution family of encoder as Normal distribution, whose mean is MLP

$$
    y^{(i)} \sim q_{\phi}(y \mid o ) = \mathcal{N}(z; \mu_\phi(x) , \Sigma_q),\quad \text{Where, } \Sigma_q = 0.1^2 I
$$

The following MLP is to make encoder mean MLP and then sample the latent vector $y^{(i)}$

In [1]:
import torch
from torch import nn

from src import make_ou_data
from src import ObsEncoder

data = make_ou_data(32, 40, 3)

obs = data["observation"]
obs_mask = data["obs_mask"]
times = data["times"]

"""
    ObsEncoder is the encoder MLP implementing, MLP mean and reparametrized sampling.
"""
encoder = ObsEncoder(obs_dim= obs.shape[-1], embed_dim=8)

"""
    Input : observation data & mask  (32, 40, 3) + (32, 40, 3)

    Output : sampled latent vector : (32, 40, 8)
"""

sampled_latent_obs = encoder(obs, obs_mask)

B,T,D =sampled_latent_obs.shape # (32, 40, 8)

print("obs:", obs.shape)
print("obs_mask:", obs_mask.shape, obs_mask.dtype)
print("y: " , sampled_latent_obs.shape)


obs: torch.Size([32, 40, 3])
obs_mask: torch.Size([32, 40]) torch.bool
y:  torch.Size([32, 40, 8])


### Transformer 

In [ ]:
import math

"""
    Make Linear Transform function ( y = W_y(.) ) for Q, K, V
"""
y = sampled_latent_obs

q_proj = nn.Linear(D, D, bias=False).to(
    device = y.device, dtype=y.dtype
)

k_proj = nn.Linear(D,D, bias=False).to(
    device = y.device, dtype=y.dtype
)

v_proj = nn.Linear(D,D, bias=False).to(
    device = y.device, dtype=y.dtype
)

"""
    make vector Q = y * W_x =  (D) * (D,D)  for all (B,T)

    Therefore, make the same y vector -> Q, K, V  vectors with own "parameter".
"""
Q = q_proj(y)
K = k_proj(y)
V = v_proj(y)

d_k = Q.shape[-1]

"""
    Q = (B,T,D) => (B) * [ [Q]_1^T , [Q]_2^T, ... , [Q]_D^T ]
    K^T = (B,D,T) => (B) * [ [K]_1 , [K]_2, ... , [K]_D ]

    ( Q @ K^T )_{t,T} = (B) * < Q_t | K_T >

    Therefore, scores : (B, T , T),  at each batch : <Q_t | K_T >/sqrt(d_k)

"""
scores = (
    Q @ K.transpose(-2,-1)
) / math.sqrt(d_k) 

"""
    using softmax to assign "exponential weight" according to normed inner product.
"""
weights = torch.softmax(scores, dim=-1)  # (B, T, T)

"""
    H = softmax( <Q | K>/sqrt(d_k)) V
"""
attention_output = weights @ V  # (B, T, D)



In [3]:
# obs_mask의 계약 확인
if obs_mask.dtype != torch.bool:
    raise TypeError("obs_mask must be a boolean tensor.")

if obs_mask.shape != y.shape[:2]:
    raise ValueError("obs_mask must have shape (B, T).")

if not obs_mask.any(dim=-1).all().item():
    raise ValueError(
        "Each sequence must contain at least one observed key."
    )


# True = 참조하지 않을 key
key_padding_mask = ~obs_mask  # (B, T)

# 모든 query에 같은 key mask를 적용
blocked_keys = key_padding_mask.unsqueeze(1)  # (B, 1, T)

# 미관측 key 열의 score를 -inf로 변경
masked_scores = scores.masked_fill(
    blocked_keys,
    float("-inf"),
)  # (B, T, T)

# 관측된 key들 사이에서 가중치 정규화
masked_weights = torch.softmax(
    masked_scores,
    dim=-1,
)  # (B, T, T)

# 관측된 key에 대응하는 value만 반영
masked_attention_output = masked_weights @ V  # (B, T, D)